# Tutorial 3: Expert-mode parameter reference

This notebook shows every file-input expert control accepted by
`pymolfit.correct`. The complete call is deliberately explicit so
that it can be used as a reference, not because a normal correction
should specify every default.

Start with Tutorials 1 and 2. Enter expert mode only after inspecting a
baseline fit and identifying a concrete limitation such as inaccurate
wavelength alignment, an unsuitable atmosphere, or an incorrect
instrumental line-spread function.

## Important interpretation

"Every expert control" does **not** mean "enable every feature." Some
parameters select alternative data sources or mutually exclusive
wavelength models. The call below includes all names but leaves
incompatible alternatives at `None` or `False`.

Change one parameter family at a time and validate improvements on
wavelength regions that were not used to tune the fit.

In [ ]:
%matplotlib widget

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pymolfit import correct, load_spectrum

In [ ]:
candidates = (Path.cwd() / "tutorials", Path.cwd())
TUTORIAL_ROOT = next(
    (path.resolve() for path in candidates if (path / "data").is_dir()),
    None,
)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook from the PyMolFit repository or tutorials directory"
    )

INPUT = TUTORIAL_ROOT / "data" / "harps_nad_crop_air.fits"

FIT_RANGES = (
    (0.5883, 0.5907),
)
EXCLUDE_RANGES = (
    (0.58875, 0.58996),
)

spectrum = load_spectrum(
    INPUT,
    wavelength_medium="air",
).to_air().to_unit("angstrom")

## Complete `correct` call

This is a valid runnable file-input call. It explicitly shows all expert controls while
retaining scientifically conservative defaults. The input is small so
that the demonstration remains practical.

The effective configuration printed after the fit is more important
than the requested values alone: automatic settings report what
PyMolFit actually selected.

In [ ]:
result = correct(
    input_path=INPUT,
    output_path=None,
    input_format=None,
    wavelength_col=None,
    flux_col=None,
    uncertainty_col=None,
    wavelength_unit='micron',
    wavelength_medium="air",
    line_list=None,
    line_list_path=None,
    hitran_par=None,
    hitran_species=None,
    hitran_min_strength=None,
    hitran_max_lines=None,
    demo_line_list=False,
    aer_catalog='auto',
    aer_cache_dir=None,
    aer_source=None,
    aer_offline=False,
    aer_reuse_molecfit=True,
    aer_timeout_s=120.0,
    partition_table=None,
    h2o_continuum=None,
    h2o_continuum_foreign_closure=False,
    co2_continuum=None,
    o2_cia=None,
    n2_cia=None,
    cia_tables=None,
    components=None,
    physical=None,
    atmosphere=None,
    atmosphere_table=None,
    atmosphere_mode='mipas_gdas',
    mipas_profile='equ',
    gdas_profile=None,
    gdas_mode='auto',
    gdas_cache_dir=None,
    gdas_download_timeout_s=15.0,
    observatory_latitude_deg=None,
    observatory_longitude_deg=None,
    observatory_altitude_m=None,
    allow_default_observatory=False,
    airmass=1.0,
    pressure_atm=0.75,
    temperature_k=280.0,
    path_length_m=8000.0,
    pwv_mm=None,
    relative_humidity_percent=None,
    mixing_ratios=None,
    continuum_order=1,
    solve_continuum_linear='auto',
    lsf_sigma_pixels='auto',
    lsf_box_width_pixels=0.0,
    lsf_lorentz_fwhm_pixels='auto',
    lsf_variable_width='auto',
    lsf_reference_wavelength_micron=None,
    lsf_kernel_width_fwhm=3.0,
    lsf_molecfit_voigt=False,
    high_resolution_grid=True,
    high_resolution_oversampling=5.0,
    high_resolution_margin_pixels=2.0,
    high_resolution_rebin_mode='molecfit_overlap',
    radiative_transfer_grid='auto',
    radiative_transfer_step_cm=None,
    radiative_transfer_max_points=2000000,
    auto_segment=True,
    segment_size=0.01,
    line_cutoff_cm=None,
    subtract_cutoff_profile=False,
    line_taper_cm=0.0,
    line_wing_mode='lblrtm_panel',
    lblrtm_sample=4.0,
    lblrtm_alfal0=0.04,
    lblrtm_avmass_amu=36.0,
    lblrtm_hwf3=64.0,
    rayleigh=False,
    rayleigh_xrayl=1.0,
    n2_continuum=False,
    n2_continuum_xn2cn=1.0,
    o2_continuum=False,
    o2_continuum_xo2cn=1.0,
    line_margin_micron=0.01,
    min_transmission=0.03,
    fit_wavelength_shift='auto',
    fit_wavelength_polynomial=False,
    wavelength_polynomial_order=1,
    fit_segment_wavelength_shifts=False,
    fit_segment_wavelength_polynomial=False,
    segment_wavelength_polynomial_order=1,
    initial_wavelength_shift=None,
    wavelength_shift_bounds=None,
    fit_lsf_sigma='auto',
    lsf_sigma_bounds=None,
    fit_lsf_box_width=False,
    lsf_box_width_bounds=(0.0, 10.0),
    fit_lsf_lorentz_fwhm='auto',
    lsf_lorentz_fwhm_bounds=None,
    fit_ranges=FIT_RANGES,
    exclude_ranges=EXCLUDE_RANGES,
    loss='linear',
    f_scale=1.0,
    ftol=1e-10,
    xtol=1e-10,
    gtol=1e-10,
    estimate_uncertainties=False,
    product_path=None,
    product_format='ascii.ecsv',
    plot_path=None,
    show_plot=False,
)

if not result.success:
    raise RuntimeError(result.message)

## Inspect the result

Even in expert mode, judge the fit from the spectrum, transmission,
residual structure, and parameters reported by PyMolFit. A successful
optimizer termination does not by itself establish scientific quality.

In [ ]:
observed = result.spectrum.to_air().to_unit("angstrom")
corrected = result.corrected.to_air().to_unit("angstrom")
valid = observed.valid & np.isfinite(observed.flux)
scale = np.nanmedian(observed.flux[valid])

figure, axes = plt.subplots(
    2,
    1,
    figsize=(12, 7),
    sharex=True,
    height_ratios=(2, 1),
    constrained_layout=True,
)
axes[0].plot(
    observed.wavelength,
    observed.flux / scale,
    color="black",
    linewidth=0.7,
    alpha=0.55,
    label="Observed",
)
axes[0].plot(
    corrected.wavelength,
    corrected.flux / scale,
    color="tab:blue",
    linewidth=0.8,
    label="Telluric corrected",
)
axes[0].set_ylabel("Flux / median")
axes[0].legend()

axes[1].plot(
    observed.wavelength,
    result.transmission,
    color="tab:red",
    linewidth=0.8,
)
axes[1].set_xlabel("Air wavelength [Angstrom]")
axes[1].set_ylabel("Transmission")
axes[1].set_ylim(0.85, 1.01)

figure.suptitle("Expert-call result")
plt.show()

# Parameter reference

Defaults shown below are the current package defaults. `None` commonly
means automatic discovery or that an optional feature is disabled; the
description for each parameter states which interpretation applies.

## Input and basic output

| Parameter | Default | Purpose and when it is useful |
|---|---:|---|
| `input_path` | `required` | Input reduced 1D spectrum in FITS, text, CSV, or ECSV format. |
| `output_path` | `None` | Optional path for the corrected wavelength/flux ASCII spectrum; ``None`` keeps it in memory only. |
| `input_format` | `None` | ``None`` infers from the filename; explicit choices are ``txt``, ``dat``, ``csv``, ``ascii``, ``ecsv``, ``fits``, ``fit``, or ``fz``. |
| `wavelength_col` | `None` | Wavelength column name or zero-based index; ``None`` uses recognized names or the first numeric column. |
| `flux_col` | `None` | Flux column name or zero-based index; ``None`` uses recognized names or the second numeric column. |
| `uncertainty_col` | `None` | Optional uncertainty column name or zero-based index; ``None`` performs an unweighted fit. |
| `wavelength_unit` | `'micron'` | Unit of input wavelengths, such as ``micron``/``um``, ``nm``, ``angstrom``/``aa``, or ``m``. |
| `wavelength_medium` | `None` | ``air`` declares standard-air wavelengths; ``vacuum`` or ``vac`` declares vacuum wavelengths. ``None`` accepts only an unambiguous air/vacuum declaration in FITS metadata; otherwise PyMolFit stops before fitting and asks for an explicit value. ``SPECSYS`` alone is not sufficient because it describes the velocity frame, not the wavelength medium. |

## Molecular lines and catalogues

| Parameter | Default | Purpose and when it is useful |
|---|---:|---|
| `line_list` | `None` | In-memory PyMolFit ``LineList``; use this instead of ``line_list_path`` or ``hitran_par``. |
| `line_list_path` | `None` | Astropy-readable PyMolFit line-list table path; ``None`` leaves line-data resolution to another source. |
| `hitran_par` | `None` | Path to a HITRAN (High-resolution Transmission Molecular Absorption Database) ``.par`` file containing molecular line positions, strengths, pressure broadening, and lower-state energies; ``None`` normally uses the managed AER catalogue. |
| `hitran_species` | `None` | Molecule names retained from HITRAN/AER, such as ``H2O``, ``O2``, or ``CO2``; ``None`` keeps every atmospheric species with transitions in the wavelength window. |
| `hitran_min_strength` | `None` | Minimum HITRAN reference line intensity to retain, controlling whether weak molecular transitions enter the opacity calculation; ``None`` applies no intensity threshold. |
| `hitran_max_lines` | `None` | Maximum number of strongest retained lines; ``None`` keeps all lines passing the other filters. |
| `demo_line_list` | `False` | ``True`` uses a tiny synthetic test list; ``False`` uses scientific line data. Never enable this for scientific correction. |
| `aer_catalog` | `'auto'` | AER is Atmospheric and Environmental Research's LBLRTM-ready molecular line catalogue derived from HITRAN; ``auto`` discovers/downloads it, a path or artifact selects it explicitly, and ``None`` disables automatic AER data. |
| `aer_cache_dir` | `None` | Directory for managed AER catalogues and wavelength-window caches; ``None`` uses PyMolFit's user cache. |
| `aer_source` | `None` | Override archive URL or local archive path used on an AER cache miss; ``None`` uses the official configured source. |
| `aer_offline` | `False` | ``True`` forbids AER network access and requires cached data; ``False`` permits download on a cache miss. |
| `aer_reuse_molecfit` | `True` | ``True`` may reuse a verified compatible local AER/Molecfit catalogue; ``False`` uses only the managed or explicit source. |
| `aer_timeout_s` | `120.0` | Per-request AER download timeout in seconds. |
| `partition_table` | `None` | Molecular partition sums convert HITRAN reference line strengths to each atmospheric layer's temperature; provide an object/table path, or use ``None`` for packaged LBLRTM/TIPS-compatible data. |

## Continuum, CIA, and custom opacity

| Parameter | Default | Purpose and when it is useful |
|---|---:|---|
| `h2o_continuum` | `None` | H2O continuum represents broad water absorption not captured by isolated discrete lines; provide an object/coefficient path, use ``lblrtm`` for packaged coefficients, or ``None`` for automatic packaged data when physical H2O lines are present. |
| `h2o_continuum_foreign_closure` | `False` | ``True`` enables the optional MT_CKD foreign-continuum closure coefficients; ``False`` uses the normal selected continuum. |
| `co2_continuum` | `None` | CO2 continuum represents broad carbon-dioxide absorption between/under discrete lines; provide an object/coefficient path, use ``lblrtm`` for packaged coefficients, or ``None`` for automatic packaged data when physical CO2 lines are present. |
| `o2_cia` | `None` | Collision-induced absorption (CIA) is broadband absorption created during molecular collisions; provide a HITRAN O2 CIA object/file path, or ``None`` to omit this explicit table. |
| `n2_cia` | `None` | Collision-induced absorption (CIA) is broadband absorption created during molecular collisions; provide a HITRAN N2 CIA object/file path, or ``None`` to omit this explicit table. |
| `cia_tables` | `None` | Mapping from additional collision-pair names to HITRAN CIA objects or paths, adding broadband collision opacity beyond O2/N2; ``None`` adds no generic CIA tables. |
| `components` | `None` | Additional or replacement absorption-component objects; ``None`` builds components from the selected physical data. |
| `physical` | `None` | ``None`` auto-detects physical HITRAN modelling, ``True`` requires it, and ``False`` disables the layered physical-atmosphere path. |

## Atmosphere and observing geometry

| Parameter | Default | Purpose and when it is useful |
|---|---:|---|
| `atmosphere` | `None` | Explicit ``AtmosphereProfile``; when provided it overrides the atmosphere builder and cannot be combined with ``atmosphere_table``. |
| `atmosphere_table` | `None` | Astropy-readable atmosphere profile table; ``None`` builds an atmosphere from FITS metadata and the selected mode. |
| `atmosphere_mode` | `'mipas_gdas'` | Selects how the vertical pressure, temperature, humidity, and gas profile is built: ``mipas_gdas``/``mipas``/``gdas`` merges a MIPAS satellite-derived climatological reference for the full/upper atmosphere with time-and-location-specific GDAS weather in the lower atmosphere, ``standard`` builds a generic layered midlatitude atmosphere, and ``single`` uses one homogeneous layer. |
| `mipas_profile` | `'equ'` | MIPAS (Michelson Interferometer for Passive Atmospheric Sounding) supplies climatological vertical pressure, temperature, and trace-gas profiles, especially above GDAS coverage; ``equ`` selects equatorial, ``std`` midlatitude standard, ``tro`` tropical, and ``auto`` currently follows Molecfit's equatorial default. |
| `gdas_profile` | `None` | GDAS (NOAA Global Data Assimilation System) supplies observation-time/location meteorology such as pressure, height, temperature, and humidity for the lower atmosphere; provide an explicit FITS profile or use ``None`` to resolve one according to ``gdas_mode``. |
| `gdas_mode` | `'auto'` | Controls the NOAA GDAS weather profile used to replace the lower part of the MIPAS climatology: ``auto`` tries exact cached/downloaded time-local data then falls back to a monthly average, ``online`` requires exact cache/download success, ``cache`` requires exact cached data without network access, and ``average`` always uses a generic monthly profile. |
| `gdas_cache_dir` | `None` | Directory for GDAS archives and interpolated profiles; ``None`` uses ``PYMOLFIT_GDAS_CACHE`` or the default user cache. |
| `gdas_download_timeout_s` | `15.0` | Per-URL timeout in seconds for ESO GDAS downloads. |
| `observatory_latitude_deg` | `None` | Geodetic observatory latitude in degrees; ``None`` reads FITS metadata when available. |
| `observatory_longitude_deg` | `None` | Geodetic observatory longitude in degrees, positive east; ``None`` reads FITS metadata when available. |
| `observatory_altitude_m` | `None` | Observatory altitude above sea level in metres; ``None`` reads FITS metadata when available. |
| `allow_default_observatory` | `False` | ``True`` permits the Paranal fallback when geometry is missing; ``False`` raises instead of silently assuming a site. |
| `airmass` | `1.0` | Line-of-sight airmass; the default allows usable FITS airmass metadata to supply the observation value in MIPAS/GDAS mode. |
| `pressure_atm` | `0.75` | Surface pressure in atmospheres for ``single``/``standard`` or metadata fallback atmosphere construction. |
| `temperature_k` | `280.0` | Surface or single-layer temperature in kelvin for fallback atmosphere construction. |
| `path_length_m` | `8000.0` | Vertical single-layer path length in metres; used only by ``atmosphere_mode="single"``. |
| `pwv_mm` | `None` | Optional precipitable-water-vapour override in millimetres; ``None`` derives water from GDAS/MIPAS or other atmosphere inputs. |
| `relative_humidity_percent` | `None` | Optional surface relative-humidity override in percent; ``None`` uses profile or FITS information. |
| `mixing_ratios` | `None` | Optional mapping of molecule name to volume mixing ratio; supplied entries override builder defaults. |

## Continuum and instrumental LSF fitting

| Parameter | Default | Purpose and when it is useful |
|---|---:|---|
| `continuum_order` | `1` | Polynomial continuum degree per fitted segment: ``0`` constant, ``1`` linear, ``2`` quadratic, and so on. |
| `solve_continuum_linear` | `'auto'` | ``"auto"`` uses the exact linear continuum solve when ``loss="linear"`` and falls back to nonlinear continuum fitting if that trial fails; robust losses such as ``soft_l1`` select nonlinear continuum fitting immediately. ``True`` always uses the linear solve and therefore requires ``loss="linear"``; ``False`` always includes continuum coefficients in the nonlinear parameter vector. |
| `lsf_sigma_pixels` | `'auto'` | The line-spread function (LSF) describes instrumental broadening of intrinsically narrow telluric lines. ``"auto"`` first derives Gaussian sigma in detector pixels from FITS resolving-power metadata and wavelength sampling, or estimates narrow observed-feature widths when metadata is unavailable; a numeric value supplies the initial/fixed sigma directly, and ``0`` disables the Gaussian component unless it is fitted. |
| `lsf_box_width_pixels` | `0.0` | A boxcar LSF component approximates finite pixel/slit integration; this is its initial/fixed width in detector pixels, and ``0`` disables it. |
| `lsf_lorentz_fwhm_pixels` | `'auto'` | A Lorentzian LSF component represents extended instrumental wings. ``"auto"`` compares Gaussian-only and Gaussian-plus-Lorentzian models on several telluric-rich pilot regions distributed over the spectrum; a numeric value supplies a fixed/initial full width at half maximum in detector pixels, and ``0`` disables the component unless explicitly fitted. |
| `lsf_variable_width` | `'auto'` | Controls wavelength dependence of instrumental broadening. ``"auto"`` (default) compares a constant-width LSF with ``width(lambda) = width_ref * (lambda / lambda_ref)**alpha`` on distributed telluric-rich pilot regions and keeps the power law only when BIC and cross-region improvement support it. ``True`` preserves the fixed legacy rule ``alpha=1``; ``False`` keeps all LSF widths constant in detector pixels. |
| `lsf_reference_wavelength_micron` | `None` | Global reference wavelength ``lambda_ref`` in microns for the LSF width law. ``None`` uses the median wavelength of the complete input spectrum once, so separately processed orders share the same width definition. |
| `lsf_kernel_width_fwhm` | `3.0` | Half-support control for numerical LSF kernels in multiples of component FWHM; larger values retain farther wings but cost more. |
| `lsf_molecfit_voigt` | `False` | ``True`` uses Molecfit's synthetic Gaussian-plus-Lorentzian Voigt approximation; ``False`` convolves the configured components normally. |
| `fit_lsf_sigma` | `'auto'` | ``"auto"`` refines an automatically estimated ``lsf_sigma_pixels`` when telluric lines are available but keeps an explicitly numeric sigma fixed; ``True`` always fits Gaussian sigma and ``False`` always keeps the resolved value fixed. |
| `lsf_sigma_bounds` | `None` | Optional lower and upper Gaussian sigma bounds in pixels. ``None`` generates broad non-negative bounds from the automatic estimate; explicit values must be increasing and non-negative. |
| `fit_lsf_box_width` | `False` | ``True`` fits boxcar LSF width within ``lsf_box_width_bounds``; ``False`` keeps it fixed. |
| `lsf_box_width_bounds` | `(0.0, 10.0)` | Lower and upper boxcar-width bounds in pixels; values must be increasing and non-negative. |
| `fit_lsf_lorentz_fwhm` | `'auto'` | ``"auto"`` performs penalized pilot-model selection when ``lsf_lorentz_fwhm_pixels="auto"`` and otherwise keeps an explicit numeric width fixed; ``True`` forces Lorentzian fitting and ``False`` disables automatic fitting. |
| `lsf_lorentz_fwhm_bounds` | `None` | Optional lower and upper Lorentzian FWHM bounds in pixels. ``None`` generates broad non-negative bounds from the Gaussian width; explicit values must be increasing and non-negative. |

## Radiative transfer, line wings, and segmentation

| Parameter | Default | Purpose and when it is useful |
|---|---:|---|
| `high_resolution_grid` | `True` | ``True`` computes, convolves, and rebins an oversampled internal model; ``False`` evaluates directly at observed samples. |
| `high_resolution_oversampling` | `5.0` | Approximate internal samples per observed pixel when ``high_resolution_grid=True``. |
| `high_resolution_margin_pixels` | `2.0` | Extra internal-grid margin on each segment edge in observed-pixel units, reducing convolution edge effects. |
| `high_resolution_rebin_mode` | `'molecfit_overlap'` | ``integrate`` averages pixel bins, ``center`` samples centres, ``sample_average`` averages enclosed samples, ``molecfit_overlap`` uses Molecfit-style overlap weights, and ``molecfit_average`` aliases sample averaging. |
| `radiative_transfer_grid` | `'auto'` | ``auto`` uses a layer-resolved native wavenumber grid; ``model`` evaluates opacity directly on the model grid for lower cost and lower fidelity. |
| `radiative_transfer_step_cm` | `None` | Explicit native radiative-transfer spacing in inverse centimetres; ``None`` derives it from atmospheric layers and line widths. |
| `radiative_transfer_max_points` | `2000000` | Maximum native-grid samples before PyMolFit asks for segmentation or a larger explicit safety limit. |
| `auto_segment` | `True` | ``True`` splits broad spectra into shared-parameter segments automatically; ``False`` fits the input as one segment and may exceed grid limits. |
| `segment_size` | `0.01` | Maximum automatic segment width in microns; ``0.01`` equals 100 Angstrom. |
| `line_cutoff_cm` | `None` | Optional maximum Voigt-wing distance in inverse centimetres; ``None`` follows ``line_wing_mode`` defaults. |
| `subtract_cutoff_profile` | `False` | ``True`` subtracts the profile value at the cutoff before truncation; ``False`` leaves the chosen wing mode unchanged. |
| `line_taper_cm` | `0.0` | Cosine-taper width in inverse centimetres at a finite line cutoff; ``0`` disables an added taper. |
| `line_wing_mode` | `'lblrtm_panel'` | ``full`` keeps all wings; ``hard_cutoff`` truncates; ``subtracted_cutoff`` edge-subtracts; ``tapered_cutoff`` tapers; ``lblrtm_subtracted`` uses fixed LBLRTM-style subtraction; ``lblrtm_dynamic`` uses dynamic per-line domains; ``lblrtm_table`` adds table-style accumulation; ``lblrtm_panel`` uses the source-parity panel/F4 treatment. |
| `lblrtm_sample` | `4.0` | LBLRTM ``SAMPLE`` control used to derive dynamic line domains; it must be positive. |
| `lblrtm_alfal0` | `0.04` | LBLRTM ``ALFAL0`` finite-domain control; ``0`` disables the corresponding finite ALFMAX cap. |
| `lblrtm_avmass_amu` | `36.0` | Representative atmospheric molecular mass in atomic mass units used for LBLRTM layer sampling. |
| `lblrtm_hwf3` | `64.0` | Outer LBLRTM Voigt F3 domain in line half-widths. |
| `rayleigh` | `False` | ``True`` includes the LBLRTM Rayleigh-scattering branch; ``False`` omits it. |
| `rayleigh_xrayl` | `1.0` | Multiplicative LBLRTM Rayleigh coefficient scale, normally ``1``. |
| `n2_continuum` | `False` | ``True`` includes LBLRTM N2 pure-rotation, fundamental, and first-overtone continuum branches; ``False`` omits them. |
| `n2_continuum_xn2cn` | `1.0` | Multiplicative LBLRTM N2-continuum coefficient scale, normally ``1``. |
| `o2_continuum` | `False` | ``True`` includes source-backed ground-based LBLRTM O2 continuum branches; ``False`` omits them. |
| `o2_continuum_xo2cn` | `1.0` | Multiplicative LBLRTM O2-continuum coefficient scale, normally ``1``. |
| `line_margin_micron` | `0.01` | Extra line-selection margin in microns around each modelled spectral interval. |
| `min_transmission` | `0.03` | Pixels with fitted atmospheric transmission below this fraction are masked in the corrected spectrum because division cannot recover reliable flux from nearly opaque regions; it must be strictly between ``0`` and ``1``. |

## Wavelength fitting and spectral masks

| Parameter | Default | Purpose and when it is useful |
|---|---:|---|
| `fit_wavelength_shift` | `'auto'` | ``"auto"`` compares no residual correction, a constant detector-pixel offset, and a smooth linear pixel-offset trend on distributed telluric-rich pilot regions, selecting by penalized fit quality; ``True`` preserves the explicit legacy constant-micron fit and ``False`` disables automatic residual alignment. Explicit polynomial or per-segment wavelength options take precedence over ``"auto"``. |
| `fit_wavelength_polynomial` | `False` | ``True`` fits a global wavelength-correction polynomial; ``False`` disables it. Do not combine it with ``fit_wavelength_shift``. |
| `wavelength_polynomial_order` | `1` | Degree of the global wavelength-correction polynomial in normalized wavelength coordinates. |
| `fit_segment_wavelength_shifts` | `False` | ``True`` fits an independent constant wavelength offset for every automatic segment, which is useful for echelle orders or detectors with separate wavelength solutions; it cannot be combined with either global wavelength-fit option. |
| `fit_segment_wavelength_polynomial` | `False` | ``True`` fits an independent wavelength-correction polynomial for every automatic segment; use this only when line residuals show within-segment wavelength distortion, and do not combine it with global or constant per-segment wavelength fitting. |
| `segment_wavelength_polynomial_order` | `1` | Degree of each independent segment wavelength polynomial when ``fit_segment_wavelength_polynomial=True``; ``0`` is a constant offset and ``1`` also permits a linear distortion. |
| `initial_wavelength_shift` | `None` | Initial constant wavelength offset in microns; ``None`` derives a suitable initial value from FITS spectral-frame metadata. |
| `wavelength_shift_bounds` | `None` | Lower and upper coefficient bounds. ``None`` uses ``(-3, 3)`` detector pixels for automatic model selection or the legacy micron bounds for explicit wavelength models; supplied values are pixels in automatic mode and microns in explicit mode. |
| `fit_ranges` | `None` | Wavelength intervals whose observed telluric features constrain molecular columns, wavelength alignment, LSF, and continuum; supply ``((start, stop), ...)`` in microns and the declared input medium, or ``None`` to let every valid pixel influence those fitted parameters. |
| `exclude_ranges` | `None` | Wavelength intervals ignored only while estimating fit parameters, normally to protect stellar/circumstellar lines, detector defects, or saturated pixels; values are in microns/input medium and the final atmospheric correction is still evaluated there. |

## Optimizer, uncertainties, and products

| Parameter | Default | Purpose and when it is useful |
|---|---:|---|
| `loss` | `'linear'` | Controls how residuals influence the fit. ``linear`` is ordinary squared-residual least squares and is the statistically preferred default for a clean, well-masked spectrum with reliable uncertainties. ``soft_l1`` is usually appropriate for a mostly clean spectrum containing a limited number of cosmic rays, bad pixels, or unmasked spectral features because it reduces their influence without ignoring normal residuals. ``huber`` is another moderate robust loss, while ``cauchy`` and ``arctan`` suppress large outliers more strongly. Robust loss cannot repair generally poor calibration, wavelength misalignment, incorrect uncertainties, or an unsuitable atmospheric model. |
| `f_scale` | `1.0` | Residual scale separating normal points from downweighted outliers for robust losses; larger values treat more residuals as normal and smaller values reject deviations more aggressively. It has no effect for ``loss="linear"``. |
| `ftol` | `1e-10` | Positive relative cost-change tolerance for optimizer termination. |
| `xtol` | `1e-10` | Positive relative parameter-step tolerance for optimizer termination. |
| `gtol` | `1e-10` | Positive gradient-norm tolerance for optimizer termination. |
| `estimate_uncertainties` | `False` | ``True`` estimates local covariance and propagates transmission uncertainty; ``False`` skips this additional work. |
| `product_path` | `None` | Optional path for the full fit-product table containing model, transmission, masks, corrected flux, metadata, and provenance. |
| `product_format` | `'ascii.ecsv'` | Astropy table writer format used for ``product_path``, for example ``ascii.ecsv`` or ``fits``. |
| `plot_path` | `None` | Optional diagnostic-plot output path; ``None`` does not save a plot. |
| `show_plot` | `False` | ``True`` opens/displays the diagnostic plot; ``False`` only saves it when ``plot_path`` is provided. |

## Mutually exclusive choices

- Use one primary line-data route: an in-memory `line_list`,
  `line_list_path`, `hitran_par`, or the managed `aer_catalog`.
- Use one atmosphere route: an in-memory `atmosphere`, an
  `atmosphere_table`, or the automatic atmosphere builder.
- Use one wavelength-correction family: automatic/global shift,
  global polynomial, independent segment shifts, or independent
  segment polynomials.
- `solve_continuum_linear=True` requires `loss="linear"`. Robust
  losses use nonlinear continuum fitting.
- Explicit output paths are optional. The returned result already
  contains the corrected spectrum, model, transmission, masks,
  fitted parameters, and provenance.

## Practical expert workflow

1. Run the minimal automatic correction.
2. Inspect line-centre residuals, continuum structure, saturated
   pixels, and the printed effective configuration.
3. Add fit and exclusion ranges based on identifiable telluric and
   astrophysical features.
4. Adjust only the parameter family implicated by the residual
   pattern.
5. Compare on held-out wavelength regions and another observation.
6. Preserve the fit-product ECSV when the result will be used for
   scientific analysis.